# From an aerial photo to a farm map — live

**GEO-ANOM Task 1 · Knowledge-sharing workshop · 7 Oct 2026**

Everything below runs live, on this laptop, on the same code that maps all 417 registered farms in Maryland. The code is hidden; each step shows a picture.

1. What the computer is given
2. What the model "thinks"
3. Turning that into buildings, and throwing out the wrong shapes
4. **Your turn:** count the houses before the model does
5. Where it breaks: a dairy farm
6. The whole state, in Google Earth

In [ ]:
import sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
plt.rcParams.update({"font.size": 13, "axes.titlesize": 15})

from geo_anom.task1 import tiles, detect, viz
from geo_anom.task1.species import species_group, GROUPS

manifest = tiles.load_manifest()
model, device = detect.load_model()
print(f"Ready: {len(manifest)} registered farms, model loaded on {device}.")

## 1. What the computer is given

A 2 km × 2 km aerial photo around one registered farm. Each pixel is 1 m × 1 m (NAIP aerial imagery, 2023).

The photo has a fourth colour our eyes can't see: **near-infrared**. On the right, near-infrared is shown as red, so healthy plants glow red. Roofs, roads and bare soil stay grey or blue.

In [ ]:
site = tiles.find_site(manifest, "Hite Farms")
img, meta = tiles.read_tile(tiles.tile_path(site))
display(Markdown(f"**{site['farm_name']}**, {site['county']} County · registry: "
                 f"{site['headcount']:,} broiler chickens · photo taken {site['naip_datetime'][:10]}"))
viz.show_tile(img, "");

## 2. What the model "thinks"

The model is a published, open neural network (Robinson et al., 2022, Microsoft Research). It learned from about 2,600 hand-labelled photos of Delmarva poultry farms. We didn't train it; we're testing how far it carries.

For every pixel it gives a score from 0 to 1: *how much does this look like part of a building?* Bright = yes.

In [ ]:
result = detect.detect_site(site, model, device)
viz.show_model_output(result.img, result.prob, result.mask);

## 3. From blobs to buildings, and the shape check

The bright blobs become shapes. Then a **shape rule** from published research (Tulbure et al., 2024), adjusted for Maryland, throws out anything that isn't poultry-house shaped. A shape is kept only if it is:

* 500–7,500 m² in area
* 55–300 m long and 10–30 m wide
* at least 2.5× longer than it is wide

Green = kept. Red dashed = thrown out.

In [ ]:
viz.show_boxes(result, size=11);

In [ ]:
table = pd.DataFrame([{"#": n, "kept": "yes" if c["kept"] else "no",
    "length (m)": round(c["stats"]["length_m"]), "width (m)": round(c["stats"]["width_m"]),
    "area (m²)": round(c["stats"]["area_m2"]), "why thrown out": "; ".join(c["reasons"])}
    for n, c in enumerate(result.candidates)])
display(table[table.kept == "no"])

Close-up of the biggest shape that was thrown out.

**Look closely: several rejected shapes on this farm sit on real poultry houses.** Some are slivers of a roof the model only partly outlined; some are two neighbouring houses merged into one shape that's too wide. The rule removes sheds, roads and homes, but it also loses some real barns. That's why every farm gets checked by eye.

In [ ]:
if result.rejected:
    biggest = max(result.rejected, key=lambda c: c["stats"]["area_m2"])
    viz.show_candidate(result, result.candidates.index(biggest), pad_m=60);

## 4. Your turn: count the poultry houses

**How many long, narrow chicken houses do you see?** Shout out a number before we reveal the model's answer.

In [ ]:
check = detect.detect_site(tiles.find_site(manifest, "Pocomoke Farm #4"), model, device)
viz.show_area(check, size=11, title="How many poultry houses?");

Now the model's answer:

In [ ]:
viz.show_boxes(check, size=11);
display(Markdown(f"**Model: {len(check.kept)} kept, {len(check.rejected)} thrown out.**"))

**What went wrong:** the houses here sit very close together, so the model's blobs run several houses into one. The merged shape is too big and too wide, and the shape rule throws it out.

This is why we check by eye, farm by farm, and why we're asking for help labelling. A number from the model on its own isn't evidence.

## 5. Where it breaks: a dairy farm

Lester C. Jones & Sons, Kent County: about **3,300 dairy cows** on the registry.

In [ ]:
dairy = detect.detect_site(tiles.find_site(manifest, "Lester C. Jones"), model, device)
viz.show_model_output(dairy.img, dairy.prob, dairy.mask)
display(Markdown(f"**Shapes found: {len(dairy.candidates)} · buildings kept: {len(dairy.kept)}.** "
                 "The model was trained on poultry houses; wide dairy barns mostly don't register as buildings at all."))

Notice the **two dark ponds** next to the barns: they could be manure storage (one of our questions for you), and the model has no idea they're there.

None of **Maryland's 14 registered dairies** has a dairy barn detected. It isn't a setting we can tune: the model needs real examples of Maryland dairy barns, lagoons and manure tanks to learn from.

**→ This is where we need your expertise (back to slides).**

## 6. The whole state

The same steps, run on all 417 registered farms (about 9 minutes on this laptop). Barns that show up in two neighbouring photos are counted once.

In [ ]:
from geo_anom.task1.pipeline import latest_run
run = latest_run()
meta_run = json.loads((run / "run_meta.json").read_text())
farms = pd.read_csv(run / "farms.csv")
farms["group"] = farms["animal_type"].map(species_group)
summary = farms.groupby("group").agg(farms=("farm_name", "size"),
                                     farms_with_buildings=("buildings", lambda s: int((s > 0).sum())),
                                     buildings=("buildings", "sum"))
summary.index = [GROUPS[g][0] for g in summary.index]
summary = summary.sort_values("farms", ascending=False)
display(Markdown(f"**{meta_run['unique_buildings']:,} buildings** across {meta_run['tiles']} farms "
                 f"({meta_run['kept_detections']:,} detections before removing barns seen twice)."))
display(summary.astype(int))

### Open it in Google Earth

One marker shape per animal type. Click any building to see its size, the model's confidence, and which farm it belongs to. A hidden folder shows everything the shape rule threw out, with the reason.

In [ ]:
from geo_anom.task1.merge import deduplicate
from geo_anom.task1.kmz import write_kmz

demo = [result, check, dairy] + [detect.detect_site(tiles.find_site(manifest, f), model, device)
                                 for f in ("Panora", "Grand View Farm", "Sheng Lin")]
kept = [f for r in demo for f in r.features()]
rejected = [c["feature"] for r in demo for c in r.rejected]
bld = deduplicate(kept, manifest)
kmz = write_kmz(json.loads(bld.to_json(drop_id=True))["features"] + rejected,
                ROOT / "notebooks/output/workshop_farms.kmz",
                sites=[r.site for r in demo], title="GEO-ANOM workshop farms")
print("Saved", kmz.name, "-- statewide map:", (run / "buildings.kmz").relative_to(ROOT))

In [ ]:
OPEN_IN_GOOGLE_EARTH = False   # set True during the talk
if OPEN_IN_GOOGLE_EARTH:
    subprocess.run(["open", "-a", "Google Earth Pro", str(kmz)])

---
**What's solid:** poultry houses. The same answer every run; checked against an independent 2016/17 hand-labelled survey.

**What isn't yet:** dairy barns, lagoons and manure tanks; and turning buildings into nitrogen and phosphorus (being checked against your lab's 2023 AWTF report).